# OSTrack on LaSOT test (Kaggle)

Settings (right panel): Accelerator = GPU T4 x2, Internet = On (needs a phone-verified account).
Then Save Version -> Save & Run All (Commit): it runs in the background (up to 12 h) and you can close the tab.

Frames are streamed from HuggingFace video by video, so no dataset needs to be attached.
Results land in `/kaggle/working/ostrack_lasot.zip` (Output tab of the saved version).

In [ ]:
# fail fast with a clear message if the session has no internet
import socket
try:
    socket.create_connection(('github.com', 443), timeout=10).close()
except OSError:
    raise SystemExit('NO INTERNET: Settings (right panel) -> Internet = On '
                     '(needs a phone-verified Kaggle account), then run again.')


In [ ]:
import subprocess  # check=True stops the run here if cloning fails
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/MariiaHamanuk/Object-Tracking.git',
                '/kaggle/working/Object-Tracking'], check=True)
!pip install -q timm==0.6.13 easydict visdom tensorboardX jpeg4py lmdb gdown
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# OSTrack vitb_256_mae_ce_32x4_ep300 weights (official Google Drive)
!gdown 1dySfPrSk-knAEo0lTXqZ1kMk3BCvVofV -O /kaggle/working/OSTrack_ep0300.pth.tar

In [ ]:
# one process per GPU; each tracks every n-th test video
import subprocess, torch, os
os.chdir('/kaggle/working/Object-Tracking')
n = max(torch.cuda.device_count(), 1)
procs = []
for i in range(n):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i))
    log = open(f'/kaggle/working/shard{i}.log', 'w')
    procs.append(subprocess.Popen(
        ['python', 'scripts/run_ostrack_lasot.py', '--checkpoint', '/kaggle/working/OSTrack_ep0300.pth.tar',
         '--out', '/kaggle/working/results/ostrack/vitb_256_mae_ce_32x4_ep300/lasot',
         '--work', '/kaggle/tmp', '--shard', str(i), '--num-shards', str(n)],
        env=env, stdout=log, stderr=subprocess.STDOUT))
print([p.wait() for p in procs])  # exit codes; 0 = all videos done

In [ ]:
!grep -hE 'FAILED|Error|\[[0-9]+/' /kaggle/working/shard*.log | tail -n 20
!ls /kaggle/working/results/ostrack/vitb_256_mae_ce_32x4_ep300/lasot | grep -vc _
!cd /kaggle/working && zip -qr ostrack_lasot.zip results shard*.log && rm -rf Object-Tracking OSTrack_ep0300.pth.tar && ls -la